# Графики для главы 4: анализ накопленных деформаций

Полный набор кода для построения графиков из подраздела о деформациях. Все графики строятся на основе уже обученных моделей и исходного датасета.

**Требуемые файлы в `/content/`:**
- `X_strain.pkl`, `y_strain.pkl` — обучающий датасет (2438 наборов)
- `mlp_strains_baseline.pkl`, `mlp_strains_optuna.pkl` — MLP baseline и Optuna
- `pinn_strains_torch.pkl`, `pinn_strains_dde.pkl`, `pinn_strains_jax.pkl` — три PINN
- `vpinn_strains.pkl` — VPINN со слабой формой

Поскольку у задачи деформаций отдельная holdout-выборка не выделялась, парные тесты ведутся на полном множестве 2438 наборов.

## Ячейка 1. Конфигурация, загрузка данных, утилиты

In [ ]:
import pickle
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error

N_R = 20
PLANE_ORDER_STRAIN = [2, 0, 1, 3]

STRAIN_NAMES = ["eps_rr", "eps_tt", "eps_zz", "eps_rz"]
COMP_TEX = [r"$\varepsilon_{rr}$", r"$\varepsilon_{\theta\theta}$",
             r"$\varepsilon_{zz}$", r"$\varepsilon_{rz}$"]

DATA_X = "/content/X_strain.pkl"
DATA_Y = "/content/y_strain.pkl"

BUNDLE_PATHS = {
    "MLP-baseline": "/content/mlp_strains_baseline.pkl",
    "MLP-Optuna":   "/content/mlp_strains_optuna.pkl",
    "PINN-torch":   "/content/pinn_strains_torch.pkl",
    "PINN-dde":     "/content/pinn_strains_dde.pkl",
    "PINN-jax":     "/content/pinn_strains_jax.pkl",
    "VPINN":        "/content/vpinn_strains.pkl",
}

MODEL_COLORS = {
    "MLP-baseline": "#ff7f0e",
    "MLP-Optuna":   "#ffbb78",
    "PINN-torch":   "#1f77b4",
    "PINN-dde":     "#17becf",
    "PINN-jax":     "#aec7e8",
    "VPINN":        "#2ca02c",
}
MODEL_LABELS = {
    "MLP-baseline": "MLP (baseline)",
    "MLP-Optuna":   "MLP (Optuna)",
    "PINN-torch":   "PINN (PyTorch)",
    "PINN-dde":     "PINN (DeepXDE)",
    "PINN-jax":     "PINN (JAX/Flax)",
    "VPINN":        "VPINN",
}

plt.rcParams.update({
    "font.family": "serif", "font.size": 10,
    "axes.linewidth": 0.8, "axes.grid": True,
    "grid.alpha": 0.25, "grid.linewidth": 0.5,
})

with open(DATA_X, "rb") as f: X_strain = pickle.load(f)
with open(DATA_Y, "rb") as f: y_strain = pickle.load(f)

n_sets = X_strain.shape[1]
proc = X_strain[0].astype(np.float32)
r_grid = np.linspace(0.0, 1.0, N_R, dtype=np.float32)

proc_rep = np.repeat(proc, N_R, axis=0)
r_rep = np.tile(r_grid, n_sets)[:, None]
X_full = np.hstack([proc_rep, r_rep]).astype(np.float32)
y_full = np.stack([y_strain[PLANE_ORDER_STRAIN[i]].reshape(-1)
                    for i in range(4)], axis=1).astype(np.float32)

print(f"N_sets = {n_sets}")
print(f"X_full.shape = {X_full.shape}, y_full.shape = {y_full.shape}")


# ── activations ────────────────────────────────────────────────────────────
def _tanh(x): return np.tanh(x)
def _gelu(x): return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))
def _relu(x): return np.maximum(x, 0)
ACT = {"tanh": _tanh, "gelu": _gelu, "relu": _relu}


def _layer_idx(key):
    m = re.search(r"(\d+)", key)
    return int(m.group(1)) if m else 0


def _extract_torch(sd, prefix):
    wk = sorted([k for k in sd if k.startswith(prefix + ".") and k.endswith(".weight")],
                 key=_layer_idx)
    bk = sorted([k for k in sd if k.startswith(prefix + ".") and k.endswith(".bias")],
                 key=_layer_idx)
    return [np.asarray(sd[k]) for k in wk], [np.asarray(sd[k]) for k in bk]


def _fourier_features(r, B):
    """VPINN: r is standardized (N, 1), B is (n_freq,). Returns (N, 2*n_freq)."""
    proj = (r * B[None, :]) * 2 * np.pi
    return np.concatenate([np.sin(proj), np.cos(proj)], axis=-1)


def predict_strain(bundle, X_input):
    """Forward inference for any strain bundle. Returns (N, 4) in physical units."""
    mx = np.asarray(bundle["mean_X"]).reshape(-1)
    sx = np.asarray(bundle["std_X"]).reshape(-1)
    sym = np.asarray(bundle["scaler_y_mean"]).reshape(-1)
    sys_ = np.asarray(bundle["scaler_y_std"]).reshape(-1)
    X_std = ((X_input - mx) / sx).astype(np.float32)

    # VPINN: detect by presence of r_embed.B
    if "model_state_dict" in bundle and "r_embed.B" in bundle["model_state_dict"]:
        sd = bundle["model_state_dict"]
        B = np.asarray(sd["r_embed.B"])
        ff = _fourier_features(X_std[:, 5:6], B)
        h = np.concatenate([X_std[:, 0:5], ff], axis=-1)
        Ws, bs = _extract_torch(sd, "net")
        act = ACT[bundle.get("activation", "tanh")]
        for i, (W, b) in enumerate(zip(Ws, bs)):
            h = h @ W.T + b
            if i < len(Ws) - 1:
                h = act(h)
        scale = np.asarray(sd["scaler_out.scale"])
        shift = np.asarray(sd["scaler_out.shift"])
        return (h * scale + shift) * sys_ + sym

    # MLP-baseline, MLP-Optuna
    if "model_state_dict" in bundle:
        sd = bundle["model_state_dict"]
        Ws, bs = _extract_torch(sd, "net")
        act_name = bundle.get("activation",
                              bundle.get("config", {}).get("activation",
                              bundle.get("best_params", {}).get("activation", "tanh")))
        act = ACT[act_name]
        h = X_std
        for i, (W, b) in enumerate(zip(Ws, bs)):
            h = h @ W.T + b
            if i < len(Ws) - 1:
                h = act(h)
        return h * sys_ + sym

    # PINN-torch (net.*) and PINN-dde (linears.*)
    if "state_dict_np" in bundle:
        sd = bundle["state_dict_np"]
        prefix = "linears" if any("linears." in k for k in sd) else "net"
        Ws, bs = _extract_torch(sd, prefix)
        act = ACT[bundle.get("activation", "tanh")]
        h = X_std
        for i, (W, b) in enumerate(zip(Ws, bs)):
            h = h @ W.T + b
            if i < len(Ws) - 1:
                h = act(h)
        return h * sys_ + sym

    # PINN-jax (Flax nested dict)
    if "params_np" in bundle:
        params = bundle["params_np"]["params"]
        dense_keys = sorted(params.keys(), key=_layer_idx)
        Ws = [np.asarray(params[k]["kernel"]) for k in dense_keys]
        bs = [np.asarray(params[k]["bias"])   for k in dense_keys]
        act = ACT[bundle.get("activation", "tanh")]
        h = X_std
        for i, (W, b) in enumerate(zip(Ws, bs)):
            h = h @ W + b   # Flax kernel is (in, out)
            if i < len(Ws) - 1:
                h = act(h)
        return h * sys_ + sym

    raise ValueError("unknown bundle structure")


def von_mises_strain(eps):
    rr, tt, zz, rz = (eps[:, i] for i in range(4))
    return np.sqrt((2/3)*(rr**2 + tt**2 + zz**2) + (4/3)*rz**2)


def smape(yt, yp, eps=1e-8):
    yt, yp = yt.ravel(), yp.ravel()
    den = (np.abs(yt) + np.abs(yp)) / 2 + eps
    return float(100 * np.mean(np.abs(yt - yp) / den))


BUNDLES = {}
for name, path in BUNDLE_PATHS.items():
    if Path(path).exists():
        with open(path, "rb") as f:
            BUNDLES[name] = pickle.load(f)
    else:
        print(f"[skip] {name}: {path} not found")

print(f"\nЗагружено бандлов: {len(BUNDLES)}")

## Ячейка 2. Инференс и сводный рейтинг моделей

In [ ]:
predictions = {}
rows = []
for name, b in BUNDLES.items():
    y_pred = predict_strain(b, X_full)
    predictions[name] = y_pred

    r2 = r2_score(y_full.reshape(-1), y_pred.reshape(-1))
    rmse = float(np.mean([np.sqrt(mean_squared_error(y_full[:, i], y_pred[:, i]))
                           for i in range(4)]))
    mae = float(np.mean(np.abs(y_full - y_pred)))
    sm = smape(y_full, y_pred)
    vm_t, vm_p = von_mises_strain(y_full), von_mises_strain(y_pred)
    vm_r2 = r2_score(vm_t, vm_p)
    vm_rmse = float(np.sqrt(mean_squared_error(vm_t, vm_p)))
    pc_r2 = [r2_score(y_full[:, i], y_pred[:, i]) for i in range(4)]
    rows.append({
        "Model": name, "R²": r2, "RMSE": rmse,
        "VM R²": vm_r2, "VM RMSE": vm_rmse,
        "SMAPE [%]": sm, "MAE": mae,
        "R²(ε_rr)": pc_r2[0], "R²(ε_θθ)": pc_r2[1],
        "R²(ε_zz)": pc_r2[2], "R²(ε_rz)": pc_r2[3],
    })

df_rank = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
df_rank.insert(0, "Rank", range(1, len(df_rank) + 1))
with pd.option_context("display.float_format", "{:,.4f}".format,
                        "display.width", 200, "display.max_columns", None):
    print(df_rank.to_string(index=False))

print(f"\nЛучшая по RMSE: {df_rank.iloc[0]['Model']}  RMSE={df_rank.iloc[0]['RMSE']:.5f}")

## Ячейка 3. Профильные графики для одного режима

Четыре панели $\varepsilon_{rr}, \varepsilon_{\theta\theta}, \varepsilon_{zz}, \varepsilon_{rz}$ для одного набора параметров. Чёрная сплошная линия — FEM, цветные пунктирные — модели.

In [ ]:
y_set = y_full.reshape(n_sets, N_R, 4)
pred_set = {n: predictions[n].reshape(n_sets, N_R, 4) for n in BUNDLES}

rmse_per_set = np.zeros((n_sets, len(BUNDLES)))
for j, n in enumerate(BUNDLES):
    diff2 = (pred_set[n] - y_set)**2
    rmse_per_set[:, j] = np.sqrt(diff2.mean(axis=(1, 2)))
avg_rmse = rmse_per_set.mean(axis=1)
good_idx = int(np.argsort(avg_rmse)[len(avg_rmse) // 2])
bad_idx  = int(np.argsort(avg_rmse)[int(len(avg_rmse) * 0.95)])


def profile_plot(set_idx, suptitle, save_to=None):
    p = proc[set_idx]
    fig, axes = plt.subplots(1, 4, figsize=(15, 4.2), sharex=True)
    for c, ax in enumerate(axes):
        ax.plot(r_grid, y_set[set_idx, :, c], "k-", lw=2.2, label="FEM", zorder=10)
        for name in BUNDLES:
            ax.plot(r_grid, pred_set[name][set_idx, :, c], "--",
                    color=MODEL_COLORS[name], lw=1.4, alpha=0.85,
                    label=MODEL_LABELS[name])
        ax.set_xlabel(r"$r$ (нормированный)", fontsize=10)
        ax.set_ylabel(f"{COMP_TEX[c]} (безразм.)", fontsize=10)
        ax.set_title(COMP_TEX[c], fontsize=12)
        if c == 0:
            ax.legend(fontsize=8, loc="best", framealpha=0.9)
    fig.suptitle(rf"{suptitle} : $Q={p[0]*100:.1f}\%,\ k={p[1]:.2f},\ "
                  rf"\alpha={p[2]:.0f}^\circ,\ \mu={p[3]:.3f},\ v={p[4]:.0f}$ м/мин",
                  fontsize=11, y=1.02)
    plt.tight_layout()
    if save_to:
        plt.savefig(save_to, dpi=300, bbox_inches="tight")
    plt.show()


profile_plot(good_idx, "Типичный набор", "/content/profile_strain_good.png")
profile_plot(bad_idx,  "Трудный набор",   "/content/profile_strain_bad.png")

## Ячейка 4. Точечные диаграммы попадания

Для каждой модели — пятипанельная диаграмма: четыре компоненты тензора деформаций и эквивалентная деформация фон Мизеса. Видимый диапазон осей ограничен перцентилями $[0{,}1;\,99{,}9]$\% для лучшей читаемости; все метрики рассчитаны на полной выборке.

In [ ]:
titles = COMP_TEX + [r"$\varepsilon_{vM}$"]
for name in BUNDLES:
    y_pred = predictions[name]
    fig, axes = plt.subplots(1, 5, figsize=(20, 4.4))
    for c, ax in enumerate(axes):
        if c < 4:
            yt = y_full[:, c]; yp = y_pred[:, c]
        else:
            yt = von_mises_strain(y_full); yp = von_mises_strain(y_pred)
        r2 = r2_score(yt, yp)
        rm = np.sqrt(mean_squared_error(yt, yp))
        sm = smape(yt, yp)
        # Trim view to percentiles for readability
        lo = np.percentile(np.concatenate([yt, yp]), 0.1)
        hi = np.percentile(np.concatenate([yt, yp]), 99.9)
        ax.scatter(yt, yp, s=4, alpha=0.35, color=MODEL_COLORS[name])
        ax.plot([lo, hi], [lo, hi], "k--", lw=1.2)
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
        ax.set_xlabel(f"FEM  {titles[c]}", fontsize=10)
        ax.set_ylabel(f"{titles[c]} pred", fontsize=10)
        ax.set_title(f"$R^2$={r2:.4f}  RMSE={rm:.4f}\nSMAPE={sm:.1f}%", fontsize=10)
    fig.suptitle(MODEL_LABELS[name], fontsize=12, fontweight="bold", y=1.02)
    plt.tight_layout()
    safe = name.lower().replace("-", "_")
    plt.savefig(f"/content/scatter_strain_{safe}.png", dpi=300, bbox_inches="tight")
    plt.show()

## Ячейка 5. Установка scikit-posthocs

In [ ]:
%pip install -q scikit-posthocs

## Ячейка 6. Тесты Фридмана и Неменьи

Для каждой пары $(набор \times модель)$ вычисляется RMSE по 20 радиальным точкам профиля. Тест Фридмана проверяет глобальную $H_0$ об идентичности распределений RMSE; пост-хок-тест Неменьи даёт попарные $p$-значения и разности средних рангов.

In [ ]:
from scipy import stats
from scipy.stats import studentized_range
import scikit_posthocs as sp

MODELS_ORDERED = list(BUNDLES.keys())
K = len(MODELS_ORDERED)
N = n_sets

rmse_pc = np.zeros((N, K, 4), dtype=np.float32)
rmse_vm = np.zeros((N, K), dtype=np.float32)
for j, name in enumerate(MODELS_ORDERED):
    yp = predictions[name]
    for s in range(N):
        yt_s = y_full[s*N_R:(s+1)*N_R]
        yp_s = yp[s*N_R:(s+1)*N_R]
        rmse_pc[s, j] = np.sqrt(np.mean((yt_s - yp_s)**2, axis=0))
        vm_t = von_mises_strain(yt_s); vm_p = von_mises_strain(yp_s)
        rmse_vm[s, j] = np.sqrt(np.mean((vm_t - vm_p)**2))

print("── Тест Фридмана ──")
friedman = {}
for ci, cn in enumerate(STRAIN_NAMES):
    chi2, p = stats.friedmanchisquare(*[rmse_pc[:, j, ci] for j in range(K)])
    friedman[cn] = (chi2, p)
    print(f"  {cn:8s}: χ²={chi2:8.2f}  p={p:.3e}")
chi2_vm, p_vm = stats.friedmanchisquare(*[rmse_vm[:, j] for j in range(K)])
friedman["vM"] = (chi2_vm, p_vm)
print(f"  {'vM':8s}: χ²={chi2_vm:8.2f}  p={p_vm:.3e}")

def mean_ranks(M):
    ranks = np.zeros_like(M, dtype=float)
    for i in range(M.shape[0]):
        ranks[i] = stats.rankdata(M[i])
    return ranks.mean(axis=0)

mean_rank = np.zeros((K, 5))
for ci in range(4):
    mean_rank[:, ci] = mean_ranks(rmse_pc[:, :, ci])
mean_rank[:, 4] = mean_ranks(rmse_vm)
rank_df = pd.DataFrame(mean_rank, index=MODELS_ORDERED,
                        columns=STRAIN_NAMES + ["vM"])
print("\n── Средние ранги ──")
print(rank_df.round(3))

CD = (studentized_range.ppf(0.95, K, np.inf) / np.sqrt(2)) * np.sqrt(K*(K+1)/(6*N))
print(f"\n  CD (α=0.05) = {CD:.4f}")

nemenyi = {}
for ci, cn in enumerate(STRAIN_NAMES):
    df_c = pd.DataFrame(rmse_pc[:, :, ci], columns=MODELS_ORDERED)
    nemenyi[cn] = sp.posthoc_nemenyi_friedman(df_c)
nemenyi["vM"] = sp.posthoc_nemenyi_friedman(
    pd.DataFrame(rmse_vm, columns=MODELS_ORDERED)
)

print("\n── Парные сравнения по vM ──")
for i in range(K):
    for j in range(i+1, K):
        dr = abs(mean_rank[i, 4] - mean_rank[j, 4])
        p = nemenyi["vM"].iloc[i, j]
        sig = "*" if dr > CD else " "
        print(f"  {MODELS_ORDERED[i]:13s} vs {MODELS_ORDERED[j]:13s}  "
              f"|ΔR|={dr:.3f} {sig}  p={p:.4f}")

## Ячейка 7. Боксплоты per-set RMSE

Ось ординат логарифмическая, поскольку RMSE между наборами охватывает три порядка.

In [ ]:
SHORT = ["MLP\nbaseline", "MLP\nOptuna", "PINN\nPyTorch",
         "PINN\nDeepXDE", "PINN\nJAX/Flax", "VPINN"]
COLORS_LIST = [MODEL_COLORS[n] for n in MODELS_ORDERED]

fig, axes = plt.subplots(2, 2, figsize=(11, 8.5))
for k, ax in enumerate(axes.flat):
    data = [rmse_pc[:, j, k] for j in range(K)]
    bp = ax.boxplot(
        data, positions=range(K), widths=0.65,
        patch_artist=True, showmeans=True,
        meanprops=dict(marker="D", markerfacecolor="white",
                       markeredgecolor="black", markersize=5),
        medianprops=dict(color="black", linewidth=1.4),
        flierprops=dict(marker="o", markersize=3, alpha=0.4,
                        markerfacecolor="grey", markeredgecolor="grey"),
        whiskerprops=dict(color="grey", linewidth=0.8),
        capprops=dict(color="grey", linewidth=0.8),
    )
    for patch, color in zip(bp["boxes"], COLORS_LIST):
        patch.set_facecolor(color); patch.set_alpha(0.85)
        patch.set_edgecolor("black"); patch.set_linewidth(0.6)
    ax.set_yscale("log")
    ax.set_xticks(range(K))
    ax.set_xticklabels(SHORT, fontsize=8.5)
    ax.set_ylabel("RMSE на наборе (лог. шкала)", fontsize=10)
    ax.set_title(COMP_TEX[k], fontsize=12)
    chi2, p = friedman[STRAIN_NAMES[k]]
    p_str = r"$p<10^{-6}$" if p < 1e-6 else f"$p={p:.3e}$"
    ax.text(0.02, 0.97, rf"Фридман: $\chi^2={chi2:.1f}$, {p_str}",
            transform=ax.transAxes, fontsize=9, va="top",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      edgecolor="grey", alpha=0.85))
plt.tight_layout()
plt.savefig("/content/fig_strain_friedman_boxplot_no_fem.png", dpi=300, bbox_inches="tight")
plt.show()

## Ячейка 8. Матрица $p$-значений Неменьи

Четыре панели по компонентам тензора деформаций. В каждой ячейке $p$-значение Неменьи и абсолютная разность средних рангов $|\Delta R|$. Тёмные ячейки — значимые различия.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle

cmap = LinearSegmentedColormap.from_list("nemenyi", [
    (0.00, "#b2182b"), (0.01, "#d6604d"), (0.05, "#fddbc7"),
    (0.05001, "#f7f7f7"), (1.00, "#f7f7f7"),
], N=256)

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for k, ax in enumerate(axes.flat):
    pmat = nemenyi[STRAIN_NAMES[k]].values
    M = pmat.copy().astype(float); np.fill_diagonal(M, np.nan)
    im = ax.imshow(M, cmap=cmap, vmin=0, vmax=1, aspect="equal")
    for i in range(K):
        for j in range(K):
            if i == j:
                ax.text(j, i, "—", ha="center", va="center",
                        fontsize=9, color="grey"); continue
            v = pmat[i, j]
            dr = abs(mean_rank[i, k] - mean_rank[j, k])
            if v < 0.001:   txt, color = f"<.001\nΔR={dr:.2f}", "white"
            elif v < 0.01:  txt, color = f"{v:.3f}\nΔR={dr:.2f}", "white"
            elif v < 0.05:  txt, color = f"{v:.3f}\nΔR={dr:.2f}", "black"
            else:           txt, color = f"{v:.2f}\nΔR={dr:.2f}", "black"
            ax.text(j, i, txt, ha="center", va="center", fontsize=7, color=color)
    ax.set_xticks(range(K)); ax.set_yticks(range(K))
    ax.set_xticklabels(SHORT, fontsize=8)
    ax.set_yticklabels([MODEL_LABELS[n].replace(" (", "\n(")
                        for n in MODELS_ORDERED], fontsize=8)
    ax.set_title(COMP_TEX[k], fontsize=12)
    for i in range(K):
        for j in range(K):
            if i != j and pmat[i, j] < 0.05:
                ax.add_patch(Rectangle((j-0.5, i-0.5), 1, 1, fill=False,
                                        edgecolor="black", linewidth=1.2))
cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6, pad=0.02, aspect=25)
cbar.set_label("p-value (Неменьи)", fontsize=10)
plt.savefig("/content/fig_strain_nemenyi_heatmap_no_fem.png", dpi=300, bbox_inches="tight")
plt.show()

## Ячейка 9. Диаграмма критической разности по интегральной метрике vM

Модели соединены горизонтальной чертой, если разница средних рангов меньше $\mathrm{CD}_{\varepsilon}$.

In [ ]:
def cd_diagram(ax, ranks, labels, N, alpha=0.05, title=""):
    k = len(ranks)
    cd = (studentized_range.ppf(1 - alpha, k, np.inf) / np.sqrt(2)) * np.sqrt(k*(k+1)/(6*N))
    order = np.argsort(ranks)
    sorted_labels = [labels[i] for i in order]
    sorted_ranks = ranks[order]
    ax.set_xlim(0.5, k + 0.5); ax.set_ylim(0, 5.5); ax.invert_yaxis()
    ax.set_yticks([]); ax.set_xticks(np.arange(1, k + 1))
    ax.tick_params(axis="x", labelsize=10)
    for s in ("left", "right", "bottom"):
        ax.spines[s].set_visible(False)
    ax.spines["top"].set_visible(True)
    ax.set_title(title, fontsize=11, pad=10)
    ax.text(0.5, 0.92, f"CD = {cd:.3f}  (Неменьи, α={alpha})",
            transform=ax.transAxes, ha="center", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      edgecolor="grey", alpha=0.8))
    cd_y = 0.6
    ax.plot([1, 1+cd], [cd_y, cd_y], "k-", linewidth=2)
    ax.plot([1, 1], [cd_y-0.1, cd_y+0.1], "k-", linewidth=2)
    ax.plot([1+cd, 1+cd], [cd_y-0.1, cd_y+0.1], "k-", linewidth=2)
    ax.text(1 + cd/2, cd_y - 0.15, "CD", ha="center", fontsize=9)
    line_y_top = 1.5
    for i in range(k):
        rank = sorted_ranks[i]
        ax.plot([rank, rank], [line_y_top, line_y_top+0.1], color="black", linewidth=1)
    n_left = (k + 1) // 2
    for i in range(n_left):
        rank = sorted_ranks[i]; label = sorted_labels[i]
        y_text = line_y_top + 0.5 + i*0.55
        ax.plot([rank, rank, 0.7], [line_y_top, y_text, y_text],
                color="black", linewidth=0.9)
        ax.text(0.65, y_text, f"{label}  ({rank:.2f})",
                ha="right", va="center", fontsize=10)
    for i in range(n_left, k):
        rank = sorted_ranks[i]; label = sorted_labels[i]
        y_text = line_y_top + 0.5 + (k-1-i)*0.55
        ax.plot([rank, rank, k+0.3], [line_y_top, y_text, y_text],
                color="black", linewidth=0.9)
        ax.text(k+0.35, y_text, f"({rank:.2f})  {label}",
                ha="left", va="center", fontsize=10)
    used_y = []
    bar_y_start = line_y_top + 0.18
    drawn = set()
    cliques = [(i, j) for i in range(k) for j in range(i+1, k)
                if sorted_ranks[j] - sorted_ranks[i] <= cd]
    for ci, cj in sorted(cliques, key=lambda x: x[0]):
        if (ci, cj) in drawn: continue
        group = [ci]
        for jj in range(ci + 1, k):
            if sorted_ranks[jj] - sorted_ranks[ci] <= cd: group.append(jj)
        if len(group) < 2: continue
        for a in group:
            for bb in group:
                if a < bb: drawn.add((a, bb))
        x_start = sorted_ranks[group[0]]; x_end = sorted_ranks[group[-1]]
        y = bar_y_start
        while any((y == uy and not (x_end < ux_a or x_start > ux_b))
                  for uy, ux_a, ux_b in used_y):
            y += 0.15
        used_y.append((y, x_start, x_end))
        ax.plot([x_start - 0.05, x_end + 0.05], [y, y], color="black",
                linewidth=2.5, solid_capstyle="butt")


fig, ax = plt.subplots(figsize=(11, 4))
cd_diagram(ax, mean_rank[:, 4],
           [MODEL_LABELS[n] for n in MODELS_ORDERED], N=N,
           title="Диаграмма критической разности (Демшар, 2006) — "
                  "RMSE по эквивалентной деформации фон Мизеса")
plt.tight_layout()
plt.savefig("/content/fig_strain_cd_diagram_no_fem.png", dpi=300, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(4, 1, figsize=(11, 14))
for k, ax in enumerate(axes):
    cd_diagram(ax, mean_rank[:, k],
               [MODEL_LABELS[n] for n in MODELS_ORDERED], N=N,
               title=f"Компонента {COMP_TEX[k]}")
plt.tight_layout()
plt.savefig("/content/fig_strain_cd_diagram_4comp_no_fem.png", dpi=300, bbox_inches="tight")
plt.show()